# ElevenLabs realtime Colab launcher

Experimental LIVE-COLAB-PROXY-01 launcher for realtime browser audio capture + ElevenLabs realtime STT. Colab creates the single-use token, starts a local HTTP server, and displays `Open realtime frontend in a new tab` for the standalone page via Colab proxy/new tab.

This is separate from the batch transcription notebook and does not save to Google Docs, update manifest, or use speaker projects. The existing batch workflow remains independent.

Runtime is pinned to a reviewed commit and checked by SHA-256 before execution. Updates require reviewing the new commit and changing both GITHUB_REF and its expected hash.


In [ ]:
import pathlib
import hashlib
import urllib.request

GITHUB_OWNER = "Just9120"
GITHUB_REPO = "Elevenlabs-API"
GITHUB_REF = "8f622994c65649c88857219481274b50aaf4cace"  # reviewed immutable release

if len(GITHUB_REF) != 40 or any(c not in "0123456789abcdef" for c in GITHUB_REF):
    raise RuntimeError("A reviewed full commit SHA is required for Colab runtime.")

RAW_BASE = f"https://raw.githubusercontent.com/{GITHUB_OWNER}/{GITHUB_REPO}/{GITHUB_REF}"
SRC_URL = f"{RAW_BASE}/elevenlabs_realtime.py"

print(f"Using GitHub ref: {GITHUB_REF}")
print(f"elevenlabs_realtime.py URL: {SRC_URL}")

workdir = pathlib.Path('/content')
src_path = workdir / 'elevenlabs_realtime.py'
def download_verified(url, destination, expected_sha256):
    with urllib.request.urlopen(url, timeout=30) as response:
        payload = response.read(2 * 1024 * 1024 + 1)
    if hashlib.sha256(payload).hexdigest() != expected_sha256:
        raise RuntimeError("Downloaded Colab runtime does not match the reviewed release; nothing will execute.")
    destination.write_bytes(payload)

download_verified(SRC_URL, src_path, "fb665021fba915e7baa9ab8392baa96ddd12dd8cdf411568f377097cf7bfed0c")

source = src_path.read_text(encoding='utf-8')
exec_globals = {'__name__': '__main__', '__file__': str(src_path)}
exec(compile(source, str(src_path), 'exec'), exec_globals)
